## **Notebook 03 — Document Parser**
- Parse a real PDF into clean markdown using Docling (local, free, no API needed). Then compare it against naive PyMuPDF parsing to see why parser quality matters.

In [2]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

# Point this to your test PDF
PDF_PATH = Path("../data/docs/test_document.pdf")

if not PDF_PATH.exists():
    print(f"PDF not found at {PDF_PATH}")
    print("Put any PDF in data/docs/ and update PDF_PATH above")
else:
    print(f"PDF found: {PDF_PATH}")
    print(f"File size: {PDF_PATH.stat().st_size / 1024:.1f} KB")

PDF found: ../data/docs/test_document.pdf
File size: 1308.4 KB


In [3]:
import fitz  # PyMuPDF

doc = fitz.open(PDF_PATH)

naive_text = ""
for page in doc:
    naive_text += page.get_text()

doc.close()

print("=== NAIVE PyMuPDF OUTPUT (first 800 chars) ===\n")
print(naive_text[:800])
print("\n...")
print(f"\nTotal characters: {len(naive_text)}")

=== NAIVE PyMuPDF OUTPUT (first 800 chars) ===

 
HR Policy and  Guideline for BDRCS 
i 
BDRCS Human Resource Policy  
 
 
Preamble 
 
Bangladesh Red Crescent Society (BDRCS) has long been awaiting a well-documented 
Human Resource Policy. The HR Policy is one of the two outcomes of a long and 
tedious journey of over five months undertaken by many people of different levels 
representing BDRCS stakeholders and the consulting firm, Development Support Link 
(DSL) entrusted with the responsibility of implementing the assignment of developing 
it. Broadly speaking, the assignment included two tasks simultaneously, the 
development of a five-year Strategic Plan and a corresponding Human Resource Policy 
for BDRCS. Therefore, it is the request of the authors to read this document in 
conjunction with the Strategic Plan for better and instan

...

Total characters: 126931


In [4]:
from docling.document_converter import DocumentConverter

converter = DocumentConverter()

print("Parsing with Docling... (first run is slow, ~30-60s)")
result = converter.convert(str(PDF_PATH))

# Export to clean markdown — preserves headers, tables, lists
markdown_text = result.document.export_to_markdown()

print("=== DOCLING MARKDOWN OUTPUT (first 800 chars) ===\n")
print(markdown_text[:800])
print("\n...")
print(f"\nTotal characters: {len(markdown_text)}")

Parsing with Docling... (first run is slow, ~30-60s)


[INFO] 2026-03-29 11:27:19,666 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-03-29 11:27:19,673 [RapidOCR] download_file.py:60: File exists and is valid: /home/md-al-amin/My-Projects/End-to-End-Agentic-Ai-Automation-Lab/.venv/lib/python3.12/site-packages/rapidocr/models/ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-03-29 11:27:19,673 [RapidOCR] main.py:53: Using /home/md-al-amin/My-Projects/End-to-End-Agentic-Ai-Automation-Lab/.venv/lib/python3.12/site-packages/rapidocr/models/ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-03-29 11:27:19,782 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-03-29 11:27:19,784 [RapidOCR] download_file.py:60: File exists and is valid: /home/md-al-amin/My-Projects/End-to-End-Agentic-Ai-Automation-Lab/.venv/lib/python3.12/site-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-03-29 11:27:19,784 [RapidOCR] main.py:53: Using /home/md-al-amin/My-Projects/End-to-End-Agentic-Ai-Automation-Lab/.venv/lib/python3.12/site

=== DOCLING MARKDOWN OUTPUT (first 800 chars) ===

## Preamble

Bangladesh Red Crescent Society (BDRCS) has long been awaiting a well-documented Human  Resource  Policy.  The  HR  Policy  is  one  of  the  two  outcomes  of  a  long  and tedious  journey  of  over  five  months  undertaken  by  many  people  of  different  levels representing BDRCS stakeholders and the consulting firm, Development Support Link (DSL) entrusted with the responsibility of implementing the assignment of developing it. Broadly speaking, the assignment included two tasks simultaneously, the development of a five-year Strategic Plan and a corresponding Human Resource Policy for  BDRCS.  Therefore,  it  is  the  request  of  the  authors  to  read  this  document  in conjunction with the Strategic Plan for better and instant understanding and seeing the relevance.

...

Total characters: 156703


In [5]:
def count_elements(text):
    lines  = text.strip().split("\n")
    return {
        "total_lines"  : len(lines),
        "headers"      : sum(1 for l in lines if l.startswith("#")),
        "table_rows"   : sum(1 for l in lines if "|" in l),
        "empty_lines"  : sum(1 for l in lines if l.strip() == ""),
        "avg_line_len" : sum(len(l) for l in lines) // max(len(lines), 1),
    }

naive_stats   = count_elements(naive_text)
docling_stats = count_elements(markdown_text)

print(f"{'Metric':<20} {'PyMuPDF':>10} {'Docling':>10}")
print("-" * 42)
for key in naive_stats:
    label = key.replace("_", " ").title()
    print(f"{label:<20} {naive_stats[key]:>10} {docling_stats[key]:>10}")

Metric                  PyMuPDF    Docling
------------------------------------------
Total Lines                2242       1080
Headers                       0        110
Table Rows                    0        152
Empty Lines                 225        396
Avg Line Len                 55        144


In [6]:
# Look at what sections Docling found
sections = []
current_section = "intro"

for line in markdown_text.split("\n"):
    if line.startswith("## "):
        current_section = line.replace("## ", "").strip()
        sections.append(current_section)
    elif line.startswith("# "):
        current_section = line.replace("# ", "").strip()
        sections.append(current_section)

print("Sections detected by Docling:")
for i, section in enumerate(sections, 1):
    print(f"  {i}. {section}")

print(f"\nTotal sections: {len(sections)}")
print("\nThese become metadata tags on every chunk in Step 5")

Sections detected by Docling:
  1. Preamble
  2. BDRCS Human Resource Policy
  3. Abbreviations
  4. Table of Contents
  5. BDRCS HR Policy
  6. One Introduction, Background and Context
  7. 1.1 What this Policy is all about
  8. 1.2 Emergence and Legal Status of Bangladesh Red Crescent Society (BDRCS)
  9. 1.3 Present Governance Structure
  10. 1.4 Methodology
  11. 1.5 Strategic Goal of the HR Policy Guideline
  12. 1.6 Strategic Objectives of the HR Policy Guideline
  13. 1.7 Scope of the HR Policy Guideline
  14. 1.8 Functional Definitions
  15. 1.9 Enforcement and Amendment of the Document
  16. 1.10 Interpretation
  17. Two BDRCS Organizational Structure
  18. 2.1 Creation of a full-fledged HR and Administration Department
  19. 2.2 Creation of a New Research &amp; Communication Department
  20. 2.3 The Organizational Structure (Organogram) BDRCS
  21. Three HR Policy Guideline for BDRCS
  22. 3.1 Analyzing Existing Staff Strength
  23. 3.2 Types of Employees
  24. 3.3 Internal C

In [7]:
# This function is what we will import in Step 5
# Write it clean — it will move to a .py file later

from dataclasses import dataclass

@dataclass
class ParsedDocument:
    filename    : str
    markdown    : str           # full markdown text
    num_pages   : int
    file_type   : str

def parse_document(file_path: str) -> ParsedDocument:
    """
    Parse a PDF or DOCX into clean markdown.
    Uses Docling for layout-aware extraction.
    Returns a ParsedDocument dataclass.
    """
    path = Path(file_path)

    if not path.exists():
        raise FileNotFoundError(f"File not found: {file_path}")

    if path.suffix.lower() not in [".pdf", ".docx", ".txt"]:
        raise ValueError(f"Unsupported file type: {path.suffix}")

    # Plain text — no parsing needed
    if path.suffix.lower() == ".txt":
        text = path.read_text(encoding="utf-8")
        return ParsedDocument(
            filename  = path.name,
            markdown  = text,
            num_pages = 1,
            file_type = "txt"
        )

    # PDF or DOCX — use Docling
    converter = DocumentConverter()
    result    = converter.convert(str(path))
    markdown  = result.document.export_to_markdown()

    # Get page count for PDF
    num_pages = 1
    if path.suffix.lower() == ".pdf":
        doc       = fitz.open(str(path))
        num_pages = len(doc)
        doc.close()

    return ParsedDocument(
        filename  = path.name,
        markdown  = markdown,
        num_pages = num_pages,
        file_type = path.suffix.lower().strip(".")
    )


# Test it
parsed = parse_document(str(PDF_PATH))
print(f"Filename  : {parsed.filename}")
print(f"File type : {parsed.file_type}")
print(f"Pages     : {parsed.num_pages}")
print(f"Markdown  : {len(parsed.markdown)} characters")
print(f"Preview   : {parsed.markdown[:120]}...")

[INFO] 2026-03-29 11:30:52,391 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-03-29 11:30:52,413 [RapidOCR] download_file.py:60: File exists and is valid: /home/md-al-amin/My-Projects/End-to-End-Agentic-Ai-Automation-Lab/.venv/lib/python3.12/site-packages/rapidocr/models/ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-03-29 11:30:52,414 [RapidOCR] main.py:53: Using /home/md-al-amin/My-Projects/End-to-End-Agentic-Ai-Automation-Lab/.venv/lib/python3.12/site-packages/rapidocr/models/ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-03-29 11:30:52,479 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-03-29 11:30:52,485 [RapidOCR] download_file.py:60: File exists and is valid: /home/md-al-amin/My-Projects/End-to-End-Agentic-Ai-Automation-Lab/.venv/lib/python3.12/site-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-03-29 11:30:52,487 [RapidOCR] main.py:53: Using /home/md-al-amin/My-Projects/End-to-End-Agentic-Ai-Automation-Lab/.venv/lib/python3.12/site

Filename  : test_document.pdf
File type : pdf
Pages     : 55
Markdown  : 156703 characters
Preview   : ## Preamble

Bangladesh Red Crescent Society (BDRCS) has long been awaiting a well-documented Human  Resource  Policy.  ...


In [8]:
import json
from pathlib import Path

# Save to data/parsed/ so Step 5 can load it without re-parsing
output_dir  = Path("../data/parsed")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / f"{parsed.filename}.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump({
        "filename"  : parsed.filename,
        "file_type" : parsed.file_type,
        "num_pages" : parsed.num_pages,
        "markdown"  : parsed.markdown,
    }, f, indent=2, ensure_ascii=False)

print(f"Saved to: {output_path}")
print(f"File size: {output_path.stat().st_size / 1024:.1f} KB")
print()
print("=" * 45)
print("STEP 4 COMPLETE")
print("=" * 45)
print("  Docling parses PDF to markdown  : OK")
print("  Structure preserved (headers,")
print("  tables, lists)                  : OK")
print("  parse_document() function ready : OK")
print("  Output saved for Step 5         : OK")
print()
print("READY FOR STEP 5 — Chunker")

Saved to: ../data/parsed/test_document.pdf.json
File size: 154.3 KB

STEP 4 COMPLETE
  Docling parses PDF to markdown  : OK
  Structure preserved (headers,
  tables, lists)                  : OK
  parse_document() function ready : OK
  Output saved for Step 5         : OK

READY FOR STEP 5 — Chunker
